<a href="https://colab.research.google.com/github/JuanZapa7a/Medical-Image-Processing/blob/main/PIM_Challenge/PIM_Challenge_Student_Practice_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UPCT Medical Image Segmentation Challenge 2026-27
## Practice 8

**Course:** Medical Image Processing (521104007)

**Professor:** Juan Zapata

> **New** content for this practice. Copy the cells below and paste them **at the end** of your own notebook (the one you started in Practice 6) — do not repeat the previous practices, you already have them done there.

## Session Guide (2 hours per session)
| Practice | Dates (Group A / B) | Session Objective | Visual Checkpoint |
|----------|----------------------|-----------------------|-------------------|
| **P6** | 28 Oct - 2 Nov | EDA, Dataset and RLE format | 6 images with masks + RLE OK |
| **P7** | 9-11 Nov | U-Net Baseline and 1st Submission | Loss Plots + Kaggle Submission |
| ▶ **P8** | 16-18 Nov | Data Augmentation and improvement | Baseline vs Augmented comparison |
| **P9** | 23-25 Nov | Inference, Threshold and Errors | 5 normal images + 2 error cases |
| **P10** | 30 Nov-2 Dec | TTA, Final Submission and Defense | Best Dice Score + Oral Defense |

> **Golden Rule:** According to Art. 7.5 of the UPCT Evaluation Regulations, attendance and in-class Checkpoint validation are mandatory to pass the practice.


# Practice 8: Data Augmentation and Model Improvement
## Single session (16 Nov Group A / 18 Nov Group B)

### Session objectives:
1. Understand Data Augmentation techniques specific to medical images
2. Implement an augmentation pipeline with Albumentations
3. Re-train the model with augmented data
4. Compare results: Baseline vs Augmented

### Golden rules in medical imaging:
- **DO**: Small rotations (±15° or less), flips, mild elastic deformations, brightness/contrast adjustment
- **DON'T**: 90° rotations (loses anatomical orientation), aggressive crops (loss of context), extreme deformations

> **CHECKPOINT P8:** Show the professor:
> 1. Visualization of 4-6 transformations applied to the same image
> 2. Comparison table: Baseline vs Augmented (Dice Score)

## Block 8.1: Data Augmentation in Medical Imaging
### From memorization to generalization

In Block 7.3 we saw this warning sign:

| Situation | Loss | Dice | Interpretation |
|-----------|------|------|-----------------|
| train/val divergence | Train drops, Val rises | Train rises, Val drops | Sign of overfitting |

With only 546 training images, a model with millions of parameters (ResNet34 + decoder) has more than enough capacity to **memorize** the specific images it sees, instead of learning patterns that **generalize** to new images. The symptom is exactly that divergence: the model improves on train but stops improving (or gets worse) on val.

### Data Augmentation as regularization

The idea is not to get more patients or more real ultrasound images: it is to produce **artificial variations** of the images you already have, so that the model never sees exactly the same image twice.

| Without augmentation | With augmentation |
|---|---|
| The model can memorize the 437 exact image-mask pairs | Each epoch sees slightly different versions (rotated, brighter, noisy...) |
| It also learns the specific noise of those images | Only what is truly **invariant** survives (the tumor shape, not the exact brightness of that ultrasound) |

> **Key idea:** Data Augmentation does not add new information about the world — it forces the model not to rely on irrelevant details of each specific image.

### Two families of transformations

In Block 7.1 we already distinguished two types of transformation in a segmentation `Dataset`, and this is where it really matters:

| Type | Examples | Is it applied to the mask? |
|------|----------|----------------------------|
| Geometric | Flip, rotation, shift/scale, elastic transform | Yes, exactly the same as to the image |
| Photometric | Brightness, contrast, gaussian noise | No — the mask stays binary (0/1) |

If you rotate the image and not the mask (or the other way around), the model trains with misaligned pairs: it will learn to segment badly, with a Dice that plummets without any code error to expose it.

### Why not all transformations are valid in medical imaging

A 90° rotation or a vertical flip are perfectly valid transformations for cat photos, but not here:

| Transformation | Clinical problem |
|-----------------|-------------------|
| 90°/180° rotation | In breast ultrasound the probe orientation is clinically relevant; rotating like this creates images that do not correspond to any real acquisition |
| Aggressive crops | Can remove the tumor completely from the image without removing it from the mask |
| Extreme elastic deformations | A tumor deformed beyond what is physiologically possible teaches the model shapes that do not exist in reality |

The general rule: **a transformation is only valid if the result could have been a real acquisition** in the clinic. That is why rotations are limited to ±10-15° (plausible variation in the probe angle) and not to full turns.

### How a pipeline is composed in Albumentations

```python
train_augment = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=10, p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])
```

Two details that usually go unnoticed:

- Each `p=` is an **independent** probability per transformation: `p=0.5` does not mean "half of the images get transformed", it means that *that* specific transformation is applied with a 50% probability, combining with the rest.
- `A.Normalize` + `ToTensorV2` automatically do what you wrote by hand in Block 8.1 (ImageNet normalization + conversion to a `(C,H,W)` tensor). That is why from here on you will no longer repeat that code manually.

### Augment only train, never val or test

`train_augment` is applied exclusively to the training `DataLoader`. The validation one keeps using the simple preprocessing from Block 8.1.

> **Question to think about:** if you also applied augmentation to the validation set, would it still be a reliable measure of how the model generalizes to new data, or would you be fooling yourselves?

The reason is simple: val and test must reflect the real distribution the model will see in production (an ultrasound exactly as the machine captures it, not a rotated, noise-added version). If you measure on artificially augmented data, the metric stops being comparable with the Kaggle Leaderboard.

> **Question to think about:** training with augmentation adds extra variability each epoch — would you expect the model to converge as fast as the Practice 7 baseline, with the same number of epochs, or would it need more time for the improvement to show?

### Quick summary

| Concept | Main idea |
|----------|-----------------|
| Overfitting | Train improves, val stagnates or worsens — sign of memorization |
| Data Augmentation | Regularization: artificial variations that prevent memorizing irrelevant details |
| Geometric transf. | Applied equally to image and mask |
| Photometric transf. | Applied only to the image |
| Medical rules | Only transformations that could match a real acquisition |
| `p=` in Albumentations | Independent probability per transformation, not a fraction of the dataset |
| Scope | Augmentation only on train; val/test keep the simple preprocessing |

### References

1. **Shorten, C., & Khoshgoftaar, T. M. (2019).** *A survey on Image Data Augmentation for Deep Learning.* Journal of Big Data.
2. **Buslaev, A., et al. (2020).** *Albumentations: Fast and Flexible Image Augmentations.* Information.

## Task 8.1: Why Data Augmentation?
### The overfitting problem
When we train with little data (546 images), the model tends to **memorize** instead of **generalize**. This shows up as:
- Train Loss drops a lot
- Val Loss stagnates or rises
- High Train Dice, low Val Dice

### The solution: Data Augmentation
It consists of creating **artificial variations** of the training images to:
1. Increase the effective dataset size
2. Force the model to learn invariant features
3. Reduce overfitting

### Transformations specific to breast ultrasound:

| Transformation | Recommended parameter | Clinical justification |
|----------------|----------------------|----------------------|
| **Horizontal Flip** | p=0.5 | The left/right breast is symmetric |
| **Vertical Flip** | p=0.3 | Less common but valid |
| **Rotation** | ±15° | The probe can have varied angles |
| **Elastic Transform** | alpha=1, sigma=50 | Simulates tissue deformations |
| **Gaussian Noise** | var_limit=(10,50) | Simulates equipment noise |
| **Brightness/Contrast** | ±0.2 | Variations in the equipment gain |

>  **IMPORTANT**: Masks must be transformed **the same way** as the images to keep the alignment.

In [ ]:
# ============================================================
# TASK 8.1: DATA AUGMENTATION PIPELINE
# ============================================================
import albumentations as A
from albumentations.pytorch import ToTensorV2

# WRITE YOUR CODE HERE
# 1. Define an augmentation pipeline with A.Compose()
# 2. Include at least these transformations:
#    - Resize to IMG_SIZE
#    - HorizontalFlip (p=0.5)
#    - VerticalFlip (p=0.3)
#    - RandomRotate90 (p=0.3)
#    - ShiftScaleRotate (shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5)
#    - ElasticTransform (alpha=1, sigma=50, p=0.3)
#    - GaussNoise (var_limit=(10, 50), p=0.3)
#    - RandomBrightnessContrast (brightness_limit=0.2, contrast_limit=0.2, p=0.5)
#    - Normalize (mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
#    - ToTensorV2()

train_augment = A.Compose([
    # Your code here...
])

print("Augmentation pipeline defined")

## Block 8.2: Visually Verifying an Augmentation Pipeline
### Why visualize before training

A misconfigured Albumentations pipeline does not raise any error — it simply trains with silently deformed data. Discovering this after 30 training epochs (several minutes or hours of GPU) is much more expensive than discovering it by looking at a figure with 6 examples before starting. That is why, in any new augmentation pipeline, the first check is always visual, not numerical.

### Each call to the pipeline is random

`train_augment` is not a deterministic function: every time you call it on the same image, the result is different. This happens because, on each call, the pipeline:

1. Decides for each transformation whether to apply it or not, according to its `p=` (an independent coin flip per transformation).
2. If applied, it samples its parameters at random within the configured range (for example, `A.Rotate(limit=10)` picks a random angle between -10° and +10°, not always the same one).

| Call | HorizontalFlip (p=0.5) | Rotate (limit=10) |
|---------|--------------------------|---------------------|
| 1 | Not applied | +7° |
| 2 | Applied | -3° |
| 3 | Applied | +10° |

That is why, calling `train_augment` 6 times on the same input image, you get 6 different outputs — it is the expected behavior, not a bug. During real training this is exactly what you want: each epoch, each image is transformed differently.

### The pipeline ends in a normalized tensor: you have to undo it to see it

If you remember Block 8.1, `train_augment` ends with `A.Normalize(...)` and `ToTensorV2()`. That means what it returns **is not a directly viewable image**:

| Property | "Normal" image (for `plt.imshow`) | `train_augment` output |
|-----------|----------------------------------------|------------------------------|
| Shape | `(H, W, C)` | `(C, H, W)` |
| Value range | `[0, 1]` or `[0, 255]` | Centered on 0, can be negative |
| Type | `numpy.ndarray` | `torch.Tensor` |

To visualize it you must undo exactly those three steps, in reverse order:

```python
aug_img = augmented['image'].permute(1, 2, 0).numpy()          # (C,H,W) -> (H,W,C)
aug_img = aug_img * std + mean                                  # undo the ImageNet normalization
aug_img = np.clip(aug_img, 0, 1)                                 # rounding errors can go outside [0,1]
```

> **Note:** the `np.clip` is not cosmetic — without it, `matplotlib` can show strange colors or raise a warning, because after the `* std + mean` operation some pixels can end up slightly outside `[0, 1]` due to numerical precision.

### What to look at in the generated images

It is not enough to check that "something different is visible" in each version. Check specifically:

- Is the anatomy still recognizable, or is the transformation so aggressive that it no longer looks like a real ultrasound?
- Have black borders or artifacts appeared due to the padding (`border_mode`) used by the rotation or the shift when going outside the original frame?
- Does any combination of transformations (rotation + brightness + noise at once) produce an unrecognizable image, even if each individual transformation seemed reasonable?

> **Question to think about:** if, when looking at the 6 augmented images, 2 of them seem clinically unrecognizable to you (tissue cannot be distinguished from background), which part of the Block 8.1 pipeline would you review first?

### Quick summary

| Concept | Main idea |
|----------|-----------------|
| Visualize before training | Detects broken pipelines without wasting hours of training |
| Randomness | Each call resolves the `p=` again and resamples parameters; 6 calls → 6 different results |
| Denormalize | `permute` to go back to `(H,W,C)`, undo `Normalize` with `*std + mean`, and `clip` to `[0,1]` |
| What to review | Recognizable anatomy, border artifacts, overly aggressive combinations |

## Task 8.2: Visualizing the Transformations
Before re-training, let's visualize how the images look with the transformations applied.

### Instructions:
1. Load an example image from the train dataset
2. Apply the `train_augment` pipeline 6 times to the same image
3. Show the original image + 6 augmented versions in a 1x7 figure
4. Observe how they change: rotation, brightness, deformation, etc.

> **CHECKPOINT P8.1:** Show the professor the figure with the 7 images (original + 6 augmented)

In [ ]:
# ============================================================
# TASK 8.2: VISUALIZING AUGMENTATIONS
# ============================================================
import cv2
import matplotlib.pyplot as plt
import numpy as np

# WRITE YOUR CODE HERE
# 1. Load an example image (use df_train.iloc[0]['image_path'])
# 2. Create a figure with 1 row x 7 columns
# 3. Show the original image in the first column
# 4. Apply train_augment 6 times and show each result

# Your code here...

plt.tight_layout()
plt.show()

## Block 8.3: Repeating an Experiment Fairly
### Why reinitialize the model instead of continuing training

`model_aug = smp.Unet(...)` creates a **new model from scratch**, it does not continue training the `model` from Practice 7. This is not an oversight: if you kept training the already-converged baseline model, you could not attribute any Dice improvement to the augmentation — it could simply be because the model had been training for more epochs in total.

> **Key idea:** for the Baseline vs Augmented comparison to mean something, both models must start from the same starting situation (freshly initialized architecture) and differ in **one single thing**: the data they see during training.

### Control variable: change one thing, not two

This is the same principle as a controlled experiment in any experimental science:

| Element | Baseline (P7) | Augmented (P8) | Does it change? |
|----------|----------------|-------------------|-----------|
| Architecture (`smp.Unet(...)`) | ResNet34 + decoder | ResNet34 + decoder | No |
| `NUM_EPOCHS` | The same value | The same value | No |
| Optimizer and `lr` | `Adam`, `lr=1e-4` | `Adam`, `lr=1e-4` | No |
| Loss function | `DiceBCELoss` | `DiceBCELoss` | No |
| Training data | Without augmentation | With `train_augment` | **Yes — the only variable** |

If you also changed the number of epochs or the learning rate at the same time you add augmentation, and the Dice improves, you would not know whether it improved because of the augmentation or the other change. An experiment with two variables at once does not allow drawing conclusions about either of them.

### The real risk of "reusing the P7 code"

The task instruction literally says "copy the P7 training code and modify only the DataLoader". That copy-paste is exactly where the most common bug of this practice sneaks in: if you forget to rename some variable, you unintentionally overwrite the baseline results.

| Original variable (P7) | New variable (P8) | If you forget to rename... |
|---|---|---|
| `model` | `model_aug` | You keep training/evaluating the baseline model, not a new one |
| `optimizer` | `optimizer_aug` | The optimizer stays tied to the baseline model parameters |
| `train_loader` | `train_loader_aug` | You train again without augmentation, even though you think it has it |
| `train_losses`, `val_losses`, `train_dices`, `val_dices` | `..._aug` | **You overwrite the baseline lists** — in Task 8.4 you will not be able to compare anything, because baseline and augmented will be the same list |

> **Question to think about:** if in Task 8.4 your "Baseline vs Augmented" plot shows two identical lines, which of the variables in the table above would you suspect first was not renamed?

### A single run is not definitive proof

In Block 7.3 we saw that Loss and Dice can diverge (the "deceptive plateau"). The same applies here: training each configuration once gives you a data point, not statistical certainty. Small differences in the final Dice between baseline and augmented can be partly due to normal training variability (random weight initialization, batch order), not only to the augmentation itself. This does not invalidate the experiment — you just have to take it into account when interpreting a small difference as "augmentation won" or "augmentation lost".

### Quick summary

| Concept | Main idea |
|----------|-----------------|
| Reinitialize the model | Prevents the improvement from being due to "more total epochs" instead of the augmentation |
| Control variable | Everything equal between baseline and augmented except the training data |
| Risk of copying code | Forgetting to rename variables silently overwrites the baseline results |
| A single run | Small differences can be normal variability, not only an effect of the augmentation |

## Task 8.3: Re-training with Augmentation
Now we are going to re-train the model using the new DataLoader with augmentation.

### Instructions:
1. Create a new `BUSIDataset` (that allows transformations) using `train_augment` instead of the base transform
2. Create a new `DataLoader` for train (the val one can stay the same)
3. Re-initialize the U-Net model (to start from scratch)
4. Train for `NUM_EPOCHS` epochs (use the same training code from P7)
5. Save the metrics: `train_losses_aug`, `val_losses_aug`, `train_dices_aug`, `val_dices_aug`
6. Apply the same checkpointing pattern as in P7, but with `model_aug`: save the weights locally every time the Val Dice improves, and when the loop finishes also save the final checkpoint (weights + history) to Google Drive — same `CHECKPOINT_DIR` as P7, but with a different file (for example `augmented_checkpoint.pth`) so you do not overwrite the baseline one. First check whether that checkpoint already exists in Drive: if so, load it and do not retrain.
6. Apply the same checkpointing as in P7 (save the best model according to Val Dice and reload it at the end), but using `model_aug` and a different file (for example `best_model_augmented.pth`) so you do not overwrite the baseline checkpoint.

> **Hint**: You can copy the P7 training code and modify only the DataLoader

In [ ]:
# ============================================================
# TASK 8.3: RE-TRAINING WITH AUGMENTATION
# ============================================================

from torch.utils.data import DataLoader

DRIVE_CHECKPOINT_PATH_AUG = CHECKPOINT_DIR / 'augmented_checkpoint.pth'

# Update the BUSIDataset class with support for Albumentations
# WRITE YOUR CODE HERE



# WRITE YOUR CODE HERE

# 1. Create new Dataset with augmentation
# train_dataset_aug = BUSIDataset(train_df, transform=train_augment)
# train_loader_aug = DataLoader(...)

# 2. Re-initialize model
# model_aug = smp.Unet(...)
# model_aug = model_aug.to(DEVICE)

# 3. Optimizer
# optimizer_aug = optim.Adam(model_aug.parameters(), lr=1e-4)

# 4. Train (copy the P7 loop but using train_loader_aug)
# Save: train_losses_aug, val_losses_aug, train_dices_aug, val_dices_aug

# 5. Checkpointing (same as in P7, but with model_aug and DRIVE_CHECKPOINT_PATH_AUG):
#    - If DRIVE_CHECKPOINT_PATH_AUG already exists: load it (weights + history) and do not retrain
#    - If it does not exist: best_val_dice_aug = -1 ; BEST_MODEL_PATH_AUG = 'best_model_augmented.pth'
#      inside the loop, if epoch_val_dice improves, update and save model_aug.state_dict()
#      when finished: load the best local one and save the complete checkpoint to Drive

print("Training with augmentation...")

# Your code here...

print("Training with augmentation completed")

## Block 8.4: Reading a Comparison of Training Curves
### Four panels, four different questions

The 2x2 figure is not simply "four pretty plots" — each panel answers a different question, and you have to read them in that order:

| Panel | Question it answers |
|-------|--------------------------|
| Train Loss | Is the model still learning on the data it sees? |
| Train Dice | How well does each model fit the training data? |
| Val Loss | Which one generalizes better according to the loss function? |
| Val Dice | Which one generalizes better according to the metric that really matters? |

### The metric that truly decides: Val Dice

Of the four panels, **Val Dice** is the one most similar to what Kaggle will score on the test set (which you have not seen). Train Loss and Train Dice only tell you how well each model memorizes the data it already knows — a high value there guarantees nothing about new images.

> **Key idea:** if you had to keep a single number to decide which model to submit to Kaggle, it would be the best Val Dice, not the best Train Dice.

### Do not look only at the final value: look at the train-val gap

Remember Block 8.1: the symptom of overfitting is that train and val separate. That is why the most informative comparison is not "what final Val Dice does each model have?", but "what distance is there between Train Dice and Val Dice in each one?":

| Model | High Train Dice and low Val Dice (large gap) | Similar Train Dice and Val Dice (small gap) |
|--------|----------------------------------------------------|-------------------------------------------------------|
| Interpretation | Overfitting: memorizes train, does not generalize | Generalizes better, even if its absolute Dice is somewhat lower |

It is perfectly possible for the Augmented model to have a **lower** Train Dice than the Baseline (because it trains with harder and more varied data) and still be the model that generalizes best, if its train-val gap is smaller.

### Comparable axes: why fixing the same limits

The instructions ask for `axes[...].set_ylim(0, 1)` on the Dice panels. This is not aesthetic: if each subplot automatically adjusted its own Y-axis range, two curves with real differences could look visually identical (or the opposite, minimal differences could look huge) just because of matplotlib's automatic zoom. To compare honestly, both curves in the same panel must share the scale.

### A small difference is not always a victory

As we saw in Block 8.3, a single run has inherent variability. If the final Val Dice of the Augmented is, for example, 0.76 versus 0.74 for the Baseline, that 2-point difference can be due to the augmentation as much as to the randomness of that particular run. A consistent, visible difference over several epochs (not only at the last point) is a more reliable signal than comparing a single final value.

> **Question to think about:** if when you finish you see that the Augmented has a worse Val Dice than the Baseline with the same number of epochs, would you immediately discard the augmentation, or is there something in Block 8.1 (about the added difficulty of training with more variability) that suggests reviewing before concluding anything?

### Quick summary

| Concept | Main idea |
|----------|-----------------|
| Four panels | Each one answers a different question; they are not interchangeable |
| Val Dice | The metric most similar to what Kaggle evaluates |
| Train-val gap | More informative than the isolated final value; small gap = better generalization |
| Shared axes | Same `set_ylim` on both curves avoids visually misleading comparisons |
| Small difference | Can be variability of a single run, not a solid conclusion |

## Task 8.4: Baseline vs Augmented Comparison
It is time to compare the results. Let's visualize the training curves of both models side by side.

### Instructions:
1. Create a 2x2 figure with:
   - Plot 1: Train Loss (Baseline vs Augmented)
   - Plot 2: Val Loss (Baseline vs Augmented)
   - Plot 3: Train Dice (Baseline vs Augmented)
   - Plot 4: Val Dice (Baseline vs Augmented)
2. Use different colors for each model
3. Add legends, titles and grid

> **CHECKPOINT P8.2:** Show the professor the comparison figure and explain the observed differences

In [ ]:
# ============================================================
# TASK 8.4: BASELINE VS AUGMENTED COMPARISON
# ============================================================
import matplotlib.pyplot as plt

# WRITE YOUR CODE HERE
# 1. Create a 2x2 figure
# 2. Plot 1: Train Loss (train_losses vs train_losses_aug)
# 3. Plot 2: Val Loss (val_losses vs val_losses_aug)
# 4. Plot 3: Train Dice (train_dices vs train_dices_aug)
# 5. Plot 4: Val Dice (val_dices vs val_dices_aug)

# Your code here...

plt.tight_layout()
plt.show()

# Print summary
print("\n" + "="*60)
print("COMPARATIVE SUMMARY")
print("="*60)
print(f"Baseline - Best Val Dice: {max(val_dices):.4f}")
print(f"Augmented - Best Val Dice: {max(val_dices_aug):.4f}")
print(f"Improvement: {max(val_dices_aug) - max(val_dices):+.4f}")